# TriPendulum-8 Colab Pro 一键训练

按 Colab 菜单 `Runtime -> Run all` 即可从安装依赖开始，自动完成环境检查、SAC/PPO 训练、动态诊断视频、评估、转换矩阵、视频渲染和结果打包。

重点：模型文件、checkpoint、TensorBoard logs、诊断视频和评估结果默认保存到 Google Drive，避免 Colab runtime 断开后丢失。

In [ ]:
# ===== 一键参数区：按需修改这里即可 =====
USE_GOOGLE_DRIVE = True
REPO_URL = "https://github.com/XuanheGuo/TriPendulum-8.git"
REPO_BRANCH = "codex"
UPDATE_PROJECT = True
PROJECT_DIR = "/content/TriPendulum-8"

# Drive 只保存持久化产物；高频 TensorBoard event 写入本地 SSD，再定期备份。
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/TriPendulum-8-outputs"
# 重新训练时换一个从未使用过的 RUN_NAME；断线续训时保持名称不变。
RUN_NAME = "sac_colab_fresh_v2"
LOCAL_RUNTIME_DIR = "/content/TriPendulum-8-runtime"
TENSORBOARD_BACKUP_INTERVAL = 300

RUN_PPO = False
RUN_SAC = True
AUTO_RESUME = True  # 新 RUN_NAME 会从零开始；断线后用同一名称自动续训
PPO_TIMESTEPS = 300_000
SAC_TIMESTEPS = 1_000_000

# 8/8 不减少 SAC 总梯度更新数，只降低逐步调度开销。
SAC_TRAIN_FREQ = 2
SAC_GRADIENT_STEPS = 2
CHECKPOINT_SAVE_FREQ = 100_000
SAVE_REPLAY_BUFFER_AT_CHECKPOINT = False
LOAD_REPLAY_BUFFER_ON_RESUME = False  # 奖励更新后不要加载旧 replay buffer

DIAGNOSTIC_ENABLED = True
DIAGNOSTIC_EVAL_FREQ = 100_000
DIAGNOSTIC_MAX_STEPS = 800
DIAGNOSTIC_N_EVAL_EPISODES = 2
DIAGNOSTIC_MODE = "curriculum_worst"
DIAGNOSTIC_WORST_K = 2

CURRICULUM_ENABLED = True
CURRICULUM_STAGE = 4
EVAL_EPISODES_PER_GOAL = 5
TRANSITION_TRIALS = 3
FINAL_RENDER_GOAL = "UUU"


In [ ]:
# ===== 挂载 Drive、进入项目目录、创建本地与持久化目录 =====
import os, sys, textwrap, subprocess, json, pathlib, shutil
from pathlib import Path

# Colab 没有 X11 显示器，MuJoCo 视频必须使用 EGL 离屏渲染。
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

if not os.path.exists(PROJECT_DIR):
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, PROJECT_DIR], check=True)
elif UPDATE_PROJECT:
    subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', PROJECT_DIR, 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
os.chdir(PROJECT_DIR)

OUTPUT_ROOT = Path(DRIVE_OUTPUT_DIR) / RUN_NAME
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
DRIVE_RUNS_DIR = OUTPUT_ROOT / "runs"
VIDEO_DIR = OUTPUT_ROOT / "videos"
DIAGNOSTIC_VIDEO_DIR = VIDEO_DIR / "diagnostics"
EVAL_DIR = OUTPUT_ROOT / "evaluation"
CONFIG_OUT_DIR = OUTPUT_ROOT / "configs"
LOCAL_ROOT = Path(LOCAL_RUNTIME_DIR) / RUN_NAME
LOCAL_RUNS_DIR = LOCAL_ROOT / "runs"

for d in [OUTPUT_ROOT, CHECKPOINT_DIR, DRIVE_RUNS_DIR, VIDEO_DIR, DIAGNOSTIC_VIDEO_DIR, EVAL_DIR, CONFIG_OUT_DIR, LOCAL_ROOT, LOCAL_RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 恢复历史 logs 到本地，TensorBoard 可继续查看同一 RUN_NAME。
if DRIVE_RUNS_DIR.exists():
    shutil.copytree(DRIVE_RUNS_DIR, LOCAL_RUNS_DIR, dirs_exist_ok=True)

print("Project:", os.getcwd())
print("Persistent output:", OUTPUT_ROOT)
print("Local TensorBoard logs:", LOCAL_RUNS_DIR)
print("Drive TensorBoard backup:", DRIVE_RUNS_DIR)
print("Checkpoints:", CHECKPOINT_DIR)


In [ ]:
# ===== 安装依赖 =====
!python -m pip install -U pip
!pip install -r requirements.txt
# Colab 可能预装已停止维护的 gym；本项目只使用 Gymnasium。
!pip uninstall -y gym >/dev/null 2>&1 || true


In [ ]:
# ===== 检查 GPU / MuJoCo / SB3 =====
!nvidia-smi || true

import mujoco, gymnasium, stable_baselines3
print('MuJoCo:', mujoco.__version__)
print('Gymnasium:', gymnasium.__version__)
print('Stable-Baselines3:', stable_baselines3.__version__)


In [ ]:
# ===== 写入 Colab Pro 训练配置 =====
import yaml, os, shutil

with open('configs/sac.yaml', 'r') as f:
    sac_cfg = yaml.safe_load(f)
with open('configs/ppo.yaml', 'r') as f:
    ppo_cfg = yaml.safe_load(f)

colab_common = {
    'curriculum': {'enabled': CURRICULUM_ENABLED, 'stage': CURRICULUM_STAGE},
    'paths': {
        'checkpoint_dir': str(CHECKPOINT_DIR),
        'tensorboard_dir': str(LOCAL_RUNS_DIR),
    },
    'diagnostic_video': {
        'enabled': DIAGNOSTIC_ENABLED,
        'eval_freq': DIAGNOSTIC_EVAL_FREQ,
        'max_steps': DIAGNOSTIC_MAX_STEPS,
        'n_eval_episodes': DIAGNOSTIC_N_EVAL_EPISODES,
        'save_dir': str(DIAGNOSTIC_VIDEO_DIR),
        'fps': 30,
        'mode': DIAGNOSTIC_MODE,
        'worst_k': DIAGNOSTIC_WORST_K,
        'fallback_goals': ['DDD', 'UUU'],
    },
    'eval': {'enabled': False},
}

for cfg, algo_name, total_steps in [(sac_cfg, 'sac', SAC_TIMESTEPS), (ppo_cfg, 'ppo', PPO_TIMESTEPS)]:
    cfg.update(colab_common)
    algo = cfg.setdefault('algorithm', {})
    algo['total_timesteps'] = int(total_steps)
    algo['save_freq'] = int(CHECKPOINT_SAVE_FREQ)
    if algo_name == 'sac':
        algo['train_freq'] = int(SAC_TRAIN_FREQ)
        algo['gradient_steps'] = int(SAC_GRADIENT_STEPS)
        algo['save_replay_buffer'] = bool(SAVE_REPLAY_BUFFER_AT_CHECKPOINT)
    local_cfg_path = f'configs/colab_{algo_name}.yaml'
    with open(local_cfg_path, 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    shutil.copy(local_cfg_path, CONFIG_OUT_DIR / f'colab_{algo_name}.yaml')

SAC_MODEL_PATH = str(CHECKPOINT_DIR / 'sac_best.zip')
PPO_MODEL_PATH = str(CHECKPOINT_DIR / 'ppo_colab_final.zip')
MODEL_PATH = SAC_MODEL_PATH if RUN_SAC else PPO_MODEL_PATH

import glob, re

def latest_checkpoint(directory, prefix):
    candidates = []
    for checkpoint in glob.glob(str(Path(directory) / f'{prefix}_*_steps.zip')):
        match = re.search(r'_(\d+)_steps\.zip$', checkpoint)
        if match:
            candidates.append((int(match.group(1)), checkpoint))
    return max(candidates, default=(0, None))

sac_resume_step, SAC_RESUME_PATH = latest_checkpoint(CHECKPOINT_DIR, 'sac') if AUTO_RESUME else (0, None)
ppo_resume_step, PPO_RESUME_PATH = latest_checkpoint(CHECKPOINT_DIR, 'ppo') if AUTO_RESUME else (0, None)
SAC_REPLAY_BUFFER_PATH = str(CHECKPOINT_DIR / f'sac_replay_buffer_{sac_resume_step}_steps.pkl') if sac_resume_step else None
if SAC_REPLAY_BUFFER_PATH and (not LOAD_REPLAY_BUFFER_ON_RESUME or not os.path.exists(SAC_REPLAY_BUFFER_PATH)):
    SAC_REPLAY_BUFFER_PATH = None
SAC_STEPS_TO_RUN = max(0, int(SAC_TIMESTEPS) - sac_resume_step)
PPO_STEPS_TO_RUN = max(0, int(PPO_TIMESTEPS) - ppo_resume_step)

print('SAC resume:', SAC_RESUME_PATH, 'completed steps:', sac_resume_step, 'remaining:', SAC_STEPS_TO_RUN, 'replay:', SAC_REPLAY_BUFFER_PATH)
print('PPO resume:', PPO_RESUME_PATH, 'completed steps:', ppo_resume_step, 'remaining:', PPO_STEPS_TO_RUN)
print(open('configs/colab_sac.yaml').read())


In [ ]:
# ===== 环境 smoke test：随机动作、角度转换、8 个绝对目标 =====
import numpy as np
from envs.tripendulum_env import TriPendulumGoalEnv
from envs.goals import GOAL_NAMES, GOAL_ABS_ANGLES, GOAL_BINARY
from utils.angle_utils import relative_to_absolute

env = TriPendulumGoalEnv()
obs, info = env.reset(goal='UUU')
print('obs shape:', obs.shape)
print('reset info:', {k: info[k] for k in ['goal_name', 'x', 'x_max']})

for i in range(10):
    obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
    print(i, 'reward=', round(reward, 3), 'goal=', info['goal_name'], 'x=', round(info['x'], 3), 'pose=', round(info.get('r_pose', 0), 3))
    if terminated or truncated:
        obs, info = env.reset()

env.close()

q = np.array([0.1, 0.2, -0.3])
print('relative q:', q)
print('absolute theta:', relative_to_absolute(q))
for name in GOAL_NAMES:
    print(name, 'binary=', GOAL_BINARY[name], 'abs=', GOAL_ABS_ANGLES[name])


In [ ]:
# ===== 本地 TensorBoard + 后台定期备份到 Drive =====
import subprocess, os, shutil

if 'tb_backup_process' in globals() and tb_backup_process.poll() is None:
    tb_backup_process.terminate()

backup_script = f'''while true; do
  mkdir -p "{DRIVE_RUNS_DIR}"
  cp -a "{LOCAL_RUNS_DIR}/." "{DRIVE_RUNS_DIR}/" 2>/dev/null || true
  sleep {int(TENSORBOARD_BACKUP_INTERVAL)}
done'''
tb_backup_process = subprocess.Popen(['bash', '-lc', backup_script])
print('TensorBoard backup PID:', tb_backup_process.pid)
print(f'Logs are backed up every {TENSORBOARD_BACKUP_INTERVAL} seconds.')

TB_LOGDIR = str(LOCAL_RUNS_DIR)
%load_ext tensorboard
%tensorboard --logdir $TB_LOGDIR --reload_interval 10


In [ ]:
# ===== PPO baseline（自动续训） =====
if RUN_PPO and PPO_STEPS_TO_RUN > 0:
    if PPO_RESUME_PATH:
        !OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 python training/train_ppo.py --config configs/colab_ppo.yaml --total-timesteps {PPO_STEPS_TO_RUN} --save-path {PPO_MODEL_PATH} --resume-from {PPO_RESUME_PATH}
    else:
        !OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 python training/train_ppo.py --config configs/colab_ppo.yaml --total-timesteps {PPO_STEPS_TO_RUN} --save-path {PPO_MODEL_PATH}
else:
    print('跳过 PPO；已关闭或目标步数已经完成。')


In [ ]:
# ===== SAC 主训练（自动续训） =====
if RUN_SAC and SAC_STEPS_TO_RUN > 0:
    if SAC_RESUME_PATH and SAC_REPLAY_BUFFER_PATH:
        !OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 python training/train_sac.py --config configs/colab_sac.yaml --total-timesteps {SAC_STEPS_TO_RUN} --save-path {SAC_MODEL_PATH} --resume-from {SAC_RESUME_PATH} --resume-replay-buffer {SAC_REPLAY_BUFFER_PATH}
    elif SAC_RESUME_PATH:
        !OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 python training/train_sac.py --config configs/colab_sac.yaml --total-timesteps {SAC_STEPS_TO_RUN} --save-path {SAC_MODEL_PATH} --resume-from {SAC_RESUME_PATH}
    else:
        !OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 python training/train_sac.py --config configs/colab_sac.yaml --total-timesteps {SAC_STEPS_TO_RUN} --save-path {SAC_MODEL_PATH}
else:
    print('跳过 SAC；已关闭或目标步数已经完成。')

shutil.copytree(LOCAL_RUNS_DIR, DRIVE_RUNS_DIR, dirs_exist_ok=True)
print('TensorBoard logs synced to:', DRIVE_RUNS_DIR)


In [ ]:
# ===== 查看 Drive 中的动态诊断报告和视频 =====
import glob, json, os
from IPython.display import Video, display

reports = sorted(glob.glob(str(DIAGNOSTIC_VIDEO_DIR / 'diagnostic_step_*.json')))
print('diagnostic reports:', reports[-5:])
if reports:
    latest_report = reports[-1]
    with open(latest_report) as f:
        report = json.load(f)
    print('latest report:', latest_report)
    print(json.dumps(report, indent=2)[:4000])

videos = sorted(glob.glob(str(DIAGNOSTIC_VIDEO_DIR / '*.mp4')))
print('diagnostic videos:', videos[-5:])
if videos:
    display(Video(videos[-1], embed=True))


In [ ]:
# ===== 8 个目标评估（输出保存到 Drive） =====
EVAL_RESULTS_CSV = str(EVAL_DIR / 'evaluation_results.csv')
!python evaluation/evaluate.py --model {MODEL_PATH} --episodes-per-goal {EVAL_EPISODES_PER_GOAL} --output {EVAL_RESULTS_CSV}

import pandas as pd
pd.read_csv(EVAL_RESULTS_CSV)


In [ ]:
# ===== 生成 8x8 姿态切换热力图（CSV/PNG 保存到 Drive） =====
TRANSITION_CSV = str(EVAL_DIR / 'transition_success_matrix.csv')
TRANSITION_HEATMAP = str(EVAL_DIR / 'transition_success_heatmap.png')
!python evaluation/transition_matrix.py --model {MODEL_PATH} --trials {TRANSITION_TRIALS} --csv {TRANSITION_CSV} --heatmap {TRANSITION_HEATMAP}

from IPython.display import Image, display
if os.path.exists(TRANSITION_HEATMAP):
    display(Image(TRANSITION_HEATMAP))


In [ ]:
# ===== 渲染最终指定 goal 视频（保存到 Drive） =====
FINAL_VIDEO_PATH = str(VIDEO_DIR / f'final_{FINAL_RENDER_GOAL}.mp4')
!python evaluation/render_video.py --model {MODEL_PATH} --goal {FINAL_RENDER_GOAL} --output {FINAL_VIDEO_PATH}

from IPython.display import Video, display
if os.path.exists(FINAL_VIDEO_PATH):
    display(Video(FINAL_VIDEO_PATH, embed=True))


In [ ]:
# ===== 打包 Drive 中的训练结果 =====
import shutil, os, glob
archive_base = Path(DRIVE_OUTPUT_DIR) / f'{RUN_NAME}_results'
zip_path = str(archive_base) + '.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(
    str(archive_base),
    'zip',
    root_dir=str(OUTPUT_ROOT),
    base_dir='.',
)
print('Created:', zip_path)
print('Size MB:', os.path.getsize(zip_path) / 1024 / 1024)
print('All persistent outputs are under:', OUTPUT_ROOT)
